# SNA Group 14 — **BẮT ĐẦU TẠI ĐÂY** (Colab v2.1)

**Đúng môi trường:** Mở notebook này trên Google Colab và chạy từng ô `▶`. Không cần cài Python trên máy.

**Lưu ý:** Đây là pipeline **Amazon-book**; giữ nguyên `trn_mat.pkl`, `val_mat.pkl`, `tst_mat.pkl`. Không dùng lại semantic profiles/embeddings GPT-3.5 của tác giả. Kết quả mới ghi trong `experiments_llm_colab_v21/<RUN_ID>/` nên không ghi đè E1–E4 cũ.

**Trước khi chạy:** Upload ZIP v2.1 vào `MyDrive` (hoặc upload ZIP qua ô thứ 2); Drive phải có dữ liệu Amazon-book 7 `.pkl`. **Raw metadata** không có sẵn trong bộ 7 `.pkl` và cần chuẩn bị riêng. Bật GPU chỉ khi chạy các notebook huấn luyện.

**Không bấm Runtime → Run all toàn bộ notebook khởi động**: các bước có điểm dừng để kiểm tra, và các cuộc gọi API mặc định tắt. Thay vì vậy chạy từng ô từ trên xuống.

## 1. Điền đúng đường dẫn và chọn chính sách nghiên cứu

In [ ]:
# === CHỈ SỬA CÁC GIÁ TRỊ TRONG Ô NÀY ===
# Sau khi tải ZIP lên Google Drive, nhập đường dẫn của ZIP ở đây.
ZIP_ON_DRIVE = '/content/drive/MyDrive/SNA_Group14_Amazon_Gemini_Colab_v2_1.zip'
ORIGINAL_PROJECT_ROOT = '/content/drive/MyDrive/Data_SNA_Group14'
RUN_ID = 'gemini_colab_pilot_001'   # đổi ID khi thay dữ liệu/model/embedding
RAW_METADATA = '/content/drive/MyDrive/Amazon_raw/meta_Books.json.gz'  # PHẢI có file raw thật
RAW_REVIEWS  = '/content/drive/MyDrive/Amazon_raw/reviews_Books_5.json.gz'  # chỉ bắt buộc khi REVIEW_POLICY='train_pair'
REVIEW_POLICY = 'none'  # 'none' (ít rủi ro leakage) hoặc 'train_pair' (chưa xác minh cấp review)
GEMINI_MODEL = 'gemini-3.5-flash-lite'  # chỉnh đúng model hiển thị trong AI Studio của bro
EMBEDDING_PROVIDER = 'local'           # local: pilot tách biến embedding; openai: so sánh gần paper hơn
LOCAL_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_LOCAL_REQUESTS_PER_UTC_DAY = 400  # GIỚI HẠN NỘI BỘ, không phải quota Google

## 2. Mount Google Drive, giải nén pipeline và checkout đúng 2 repo

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, os, zipfile, shutil, hashlib

drive.mount('/content/drive')
ZIP = Path(ZIP_ON_DRIVE)
if not ZIP.is_file():
    print('Không thấy ZIP ở Drive, chọn ZIP từ máy tính để tải vào Colab (chỉ phiên hiện tại):')
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1 and next(iter(uploaded)).endswith('.zip'), 'Chỉ upload 1 ZIP của pipeline v2.1'
    ZIP=Path('/content')/next(iter(uploaded))
    assert ZIP.is_file()

TMP = Path('/content/sna_group14_colab_v21')
TMP.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP) as archive:
    root=TMP.resolve()
    for m in archive.infolist():
        target=(TMP/m.filename).resolve()
        assert target==root or target.is_relative_to(root), 'Unsafe ZIP entry: '+m.filename
    archive.extractall(TMP)
PIPELINE=TMP/'SNA_Group14_Amazon_Gemini_Colab_v2_1'
assert (PIPELINE/'run.py').is_file(), 'Sai ZIP, thiếu run.py v2.1'
assert (PIPELINE/'sna/experiments.py').is_file()
print('PIPELINE LOCAL:',PIPELINE)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PIPELINE/'requirements.txt')],check=True)

def git_checkout(url, folder, commit):
    repo=Path(folder)
    if not repo.exists():
        subprocess.run(['git','clone',url,str(repo)],check=True)
        subprocess.run(['git','-C',str(repo),'checkout','--detach',commit],check=True)
    assert (repo/'.git').exists(), f'Not a git checkout: {repo}'
    head=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    assert head==commit, f'Git SHA khác bản đã kiểm kê: {repo} -> {head}; dùng runtime mới, không tự reset.'
    return repo

PAPER_SHA='22413752246de3dee8ab0d509f7f7a8889080f95'
GROUP_SHA='c17caeb4448a8add7939510411339af918d258bb'
git_checkout('https://github.com/HKUDS/RLMRec.git','/content/RLMRec',PAPER_SHA)
git_checkout('https://github.com/trungkien1502/project_sna_group14.git','/content/project_sna_group14',GROUP_SHA)
print('Pinned source repos: PASS')

## 3. Tạo config của nhánh chạy mới (không chứa API keys)

In [ ]:
# Tạo config riêng trên Drive; không ghi API key xuống tệp.
RUN_ROOT=Path(ORIGINAL_PROJECT_ROOT)/'experiments_llm_colab_v21'
BRANCH=RUN_ROOT/RUN_ID
assert str(BRANCH).startswith(str(Path(ORIGINAL_PROJECT_ROOT)/'experiments_llm_colab_v21')+'/')
CFG=BRANCH/'settings/colab_config.json'
proposed=json.loads((PIPELINE/'configs/amazon_v2.example.json').read_text(encoding='utf-8'))
proposed['run_id']=RUN_ID
proposed['paths'].update({
    'project_root':ORIGINAL_PROJECT_ROOT,
    'run_root':str(RUN_ROOT),
    'rlmrec_repo':'/content/RLMRec',
    'group_repo':'/content/project_sna_group14',
    'raw_metadata':RAW_METADATA,
    'raw_reviews':RAW_REVIEWS})
proposed['profiles'].update({
    'model':GEMINI_MODEL,'user_review_policy':REVIEW_POLICY,
    'max_daily_requests':MAX_LOCAL_REQUESTS_PER_UTC_DAY})
proposed['embedding'].update({'provider':EMBEDDING_PROVIDER,
                               'local_model':LOCAL_EMBEDDING_MODEL})
CFG.parent.mkdir(parents=True,exist_ok=True)
if CFG.is_file():
    prior=json.loads(CFG.read_text(encoding='utf-8'))
    def scientific_fields(z):
        return dict(paths={k:v for k,v in z['paths'].items() if k not in ('run_root',)},
                    source=z['source'],data=z['data'],
                    profiles={k:v for k,v in z['profiles'].items() if k not in ('max_daily_requests','min_interval_seconds','max_retries')},
                    embedding=z['embedding'])
    assert scientific_fields(prior)==scientific_fields(proposed), 'Config nghiên cứu của RUN_ID này đã thay đổi: chọn RUN_ID mới'
else:
    CFG.write_text(json.dumps(proposed,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('CONFIG:',CFG)
print('NEW BRANCH:',BRANCH)
print('Original data remains:',ORIGINAL_PROJECT_ROOT)

def run(*args):
    cmd=[sys.executable,str(PIPELINE/'run.py'),'--config',str(CFG),*map(str,args)]
    print('\n==>', ' '.join(args),flush=True)
    subprocess.run(cmd,cwd=PIPELINE,check=True)

## 4. Kiểm tra ma trận train/validation/test và mapper trước khi làm gì khác

In [ ]:
run('audit')

## 5. Xử lý raw Amazon metadata/reviews

Phải đưa raw metadata khớp ID ASIN của release mà RLMRec dùng vào Drive. Nếu để `REVIEW_POLICY='none'`, code **không đọc review text**; nếu `train_pair`, mỗi review phải khớp một train user–item pair, nhưng **vẫn không chứng minh được tính train-only ở cấp sự kiện**. `index-raw` sẽ từ chối dữ liệu lệch mapper hoặc độ bao phủ thấp.

In [ ]:
assert Path(RAW_METADATA).is_file(), ('Chưa tìm được raw metadata: '+RAW_METADATA+' — hãy tải đúng release Amazon về Drive, kiểm tra ASIN và sửa ô cấu hình')
if REVIEW_POLICY=='train_pair':
    assert Path(RAW_REVIEWS).is_file(), 'Review mode train_pair cần raw review file phù hợp'
run('index-raw')

## 6. Tạo item prompts từ title/description (chưa gọi Gemini)

In [ ]:
run('build-prompts','--kind','item')
run('profiles','--kind','item','--dry-run','--limit','2')

## 7. Gọi Gemini thử **3 item** (có thể tiêu tốn quota/chi phí)

Đổi `DO_ITEM_PILOT=True` để bắt đầu. API key lấy từ Colab Secrets `GEMINI_API_KEY` hoặc nhập bằng getpass, **không lưu vào JSON hoặc notebook**. Sau pilot, đọc kỹ `profiles/item/0.json` để quyết định có chạy hàng loạt.

In [ ]:
DO_ITEM_PILOT=False
if DO_ITEM_PILOT:
    from getpass import getpass
    try:
        from google.colab import userdata
        api_key=userdata.get('GEMINI_API_KEY')
    except Exception:
        api_key=getpass('Gemini API key (hidden): ')
    assert api_key, 'Missing API key'
    os.environ['GEMINI_API_KEY']=api_key
    os.environ['SNA_ALLOW_REMOTE_LLM']='YES'
    run('profiles','--kind','item','--limit','3')
else:
    print('Item pilot OFF — đổi DO_ITEM_PILOT=True nếu muốn dùng quota API')

## 8. Tạo đủ **9.332 item profiles** trước khi sang user

`--limit 300` chỉ tạo tối đa 300 hồ sơ **mới** và sẽ bỏ qua hồ sơ đã hoàn tất. Hết quota thì dừng, ngày sau chạy tiếp cùng RUN_ID. Kiểm tra chi phí/quota thực tế trong Google AI Studio.

In [ ]:
DO_ITEM_BATCH=False
if DO_ITEM_BATCH:
    assert os.environ.get('SNA_ALLOW_REMOTE_LLM')=='YES', 'Kết nối Gemini API ở ô 7 trước'
    run('profiles','--kind','item','--limit','300')
else:
    print('Item batch OFF')

## 9. Tạo user prompts — phải có đủ item profiles mới

In [ ]:
DO_BUILD_USER_PROMPTS=False
if DO_BUILD_USER_PROMPTS:
    run('build-prompts','--kind','user')
    run('profiles','--kind','user','--dry-run','--limit','2')
else:
    print('Chưa build user prompts; chỉ bật sau khi item profiles hoàn tất')

## 10. Gọi Gemini cho user profiles — thử nhỏ rồi chạy nhiều phiên

In [ ]:
DO_USER_PILOT=False
DO_USER_BATCH=False
if DO_USER_PILOT or DO_USER_BATCH:
    assert os.environ.get('SNA_ALLOW_REMOTE_LLM')=='YES', 'Kết nối Gemini API ở ô 7 trước'
    if DO_USER_PILOT:run('profiles','--kind','user','--limit','3')
    if DO_USER_BATCH:run('profiles','--kind','user','--limit','300')
else:
    print('User profile calls OFF')

## 11. Tạo embeddings theo **một điều kiện nghiên cứu đã chọn**

- `openai` + `text-embedding-ada-002` giữ phương pháp embedding gần bản paper (cần API/billing).
- `local` (mặc định pilot) đổi cả embedding model ⇒ **không thể quy chênh lệch metrics chỉ cho Gemini**.

Đổi `DO_EMBED=True` khi đủ 20.332 profiles.

In [ ]:
DO_EMBED=False
if DO_EMBED:
    if EMBEDDING_PROVIDER=='openai':
        from getpass import getpass
        os.environ['OPENAI_API_KEY']=os.environ.get('OPENAI_API_KEY') or getpass('OpenAI embedding key (hidden): ')
        os.environ['SNA_ALLOW_REMOTE_EMBEDDING']='YES'
        subprocess.run([sys.executable,'-m','pip','install','-q','openai'],check=True)
    elif EMBEDDING_PROVIDER=='local':
        subprocess.run([sys.executable,'-m','pip','install','-q','sentence-transformers'],check=True)
    else:raise ValueError('Only openai/local for real research')
    run('embed','--kind','item')
    run('embed','--kind','user')
else:
    print('Embedding generation OFF')

## 12. Đóng gói đủ 7 file cho RLMRec và kiểm tra training preflight

Không chạy bước này nếu profiles/embeddings chưa hoàn tất. Ba ma trận split sẽ được copy giữ SHA256; **không dùng pickle profiles cũ**.

In [ ]:
DO_PACKAGE=False
if DO_PACKAGE:
    run('package')
    run('preflight')
    run('plan')
else:
    print('Package OFF. Chỉ bật sau khi item/user embeddings được tạo xong.')

## 13. Chuẩn bị notebook E1–E4 (chạy native Colab)

Sau khi package PASS, mở notebook `02_PREPARE_STAGE_Colab.ipynb` trong ZIP để chọn E1/E2/E3A–E3D/E4. Mỗi lượt tạo một `.ipynb` vào Drive ở đường dẫn `<RUN_ID>/runtime/notebooks/`. Mở notebook đó bằng Colab và bấm **Runtime → Run all**, theo đúng thứ tự và kiểm tra PASS sau mỗi stage.

**Không sử dụng** `run-all --execute`: chức năng tự chạy bằng Jupyter kernel phụ bị khóa để tránh lỗi Colab Drive và tái sử dụng dữ liệu cũ.